### 1- یک سیستم برای تحلیل احساس کامنت کاربر ها
### 3- مشتری ها از چه کالاهایی راضی تر و از چه کالاهایی نارضایتی زیادی داشتند؟

### import data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
data_path = os.getenv('DATA_PATH')
products = pd.read_csv(f'{data_path}/BaSalam.products.csv')
reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv')
labeled_comments = pd.read_csv(f"{data_path}/all_labeled_comments.csv")
map_category = pd.read_csv(f"{data_path}/category_map.csv")
category_mapping = dict(zip(map_category['sub_category'], map_category['main_category']))
products['main_category'] = products['categoryTitle'].map(category_mapping).fillna("سایر")
products[['categoryTitle','main_category']].sample()

### 1- 

#### parsbert

In [ ]:
# load the model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
save_directory = "../models/HooshvareLab/sentiment-snappfood//"

tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForSequenceClassification.from_pretrained(save_directory)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    labels = ["مثبت", "منفی"]
    
    predicted_class = torch.argmax(probs).item()
    predicted_label = labels[predicted_class]

    return predicted_label, probs.tolist()[0]

text = "افتضاح"
label, probabilities = predict_sentiment(text)
print(f"احساسات متن: {label}, احتمال‌ها: {probabilities}")

In [ ]:
for i in reviews[reviews['description'].notna()]['description'].sample(5).values:
    label, probabilities = predict_sentiment(i)
    print(i)
    print(f"احساسات متن: {label}, احتمال‌ها: {probabilities}", end='\n\n')

### 2-

#### stars

In [ ]:
filtered_products = reviews.groupby('productId').filter(lambda x: len(x) > 5)
product_with_high_satisfication_id = filtered_products.groupby('productId')['star'].agg(['mean', 'count']
                    ).sort_values(by=['mean', 'count'], ascending=[False, False]).head(1000).index.tolist()
product_with_low_satisfication_id = filtered_products.groupby('productId')['star'].agg(['mean', 'count']
                    ).sort_values(by=['mean', 'count'], ascending=[True, False]).head(1000).index.tolist()
proudct_with_high_satisfication = products[products['_id'].isin(product_with_high_satisfication_id)]
proudct_with_low_satisfication = products[products['_id'].isin(product_with_low_satisfication_id)]
proudct_with_high_satisfication.groupby('categoryTitle').size().sort_values(ascending=False).head(5).reset_index()
proudct_with_low_satisfication.groupby('categoryTitle').size().sort_values(ascending=False).head(5).reset_index()
high_data = proudct_with_high_satisfication.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
high_data['precent'] = round(high_data['count']/sum(high_data['count']), 3)
low_data = proudct_with_low_satisfication.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
low_data['precent'] = round(low_data['count']/sum(low_data['count']), 3)
data = products.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
data['precent'] = round(data['count']/sum(data['count']), 3)
data

In [ ]:
all_products_data = products.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
all_products_data['percent'] = round(all_products_data['count'] / sum(all_products_data['count']), 3)

high_satisfaction_data = proudct_with_high_satisfication.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
high_satisfaction_data['percent'] = round(high_satisfaction_data['count'] / sum(high_satisfaction_data['count']), 3)

low_satisfaction_data = proudct_with_low_satisfication.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
low_satisfaction_data['percent'] = round(low_satisfaction_data['count'] / sum(low_satisfaction_data['count']), 3)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=all_products_data['main_category'],
    y=all_products_data['percent'],
    name='All Products',
    marker_color='blue',
    opacity=0.6,
    text=all_products_data['count'], 
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=high_satisfaction_data['main_category'],
    y=high_satisfaction_data['percent'],
    name='High Satisfaction Products',
    marker_color='green',
    opacity=0.6,
    text=high_satisfaction_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=low_satisfaction_data['main_category'],
    y=low_satisfaction_data['percent'],
    name='Low Satisfaction Products',
    marker_color='red',
    opacity=0.6,
    text=low_satisfaction_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.update_layout(
    title='Percentage of All Products and High Satisfaction Products by Main Category',
    xaxis_title='Main Category',
    yaxis_title='Percent',
    barmode='group',
    xaxis_tickangle=-30  
)
fig.show()

In [ ]:
fig = px.pie(high_satisfaction_data, values='percent', names='main_category', title='Percentage of High Satisfaction Products by Main Category', hover_data=['count'])
fig.show()

In [ ]:
fig = px.pie(low_satisfaction_data, values='percent', names='main_category', title='Percentage of Low Satisfaction Products by Main Category', hover_data=['count'])
fig.show()

#### product rating_average col

In [ ]:
products_ = products[products['rating_average'] !=  0]
high_satisfaction_data = products_[['main_category','rating_average']].sort_values(by='rating_average',ascending=False
                                                                            ).head(500000)
low_satisfaction_data = products_[['main_category','rating_average']].sort_values(by='rating_average',ascending=True
                                                                            ).head(20000)
high_data = high_satisfaction_data['main_category'].value_counts().reset_index()
high_data['percent'] = high_data['count']/sum(high_data['count'])
low_data = low_satisfaction_data['main_category'].value_counts().reset_index()
low_data['percent'] = low_data['count']/sum(low_data['count'])

In [ ]:
all_products_data = products.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
all_products_data['percent'] = round(all_products_data['count'] / sum(all_products_data['count']), 3)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=all_products_data['main_category'],
    y=all_products_data['percent'],
    name='All Products',
    marker_color='blue',
    opacity=0.6,
    text=all_products_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=high_data['main_category'],
    y=high_data['percent'],
    name='High satisfaction Products',
    marker_color='green',
    opacity=0.6,
    text=high_data['count'], 
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.add_trace(go.Bar(
    x=low_data['main_category'],
    y=low_data['percent'],
    name='Low satisfaction Products',
    marker_color='red',
    opacity=0.6,
    text=low_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.show()

#### labeled comments

In [ ]:
map_label = {
    'Positive':2,
    'Neutral':1,
    'Negative':0   
}
labeled_comments['sentiment_by_trained_model'] = labeled_comments['sentiment_by_trained_model'].map(map_label)
labeled_comments_categories = reviews[reviews['_id'].isin(labeled_comments['_id'].values.tolist())]
labeled_comments_ = pd.merge(labeled_comments, reviews[['_id','productId']], left_on='_id', right_on='_id')
labeled_comments_ = pd.merge(labeled_comments_, products[['_id', 'main_category',]], left_on='productId', right_on='_id')
labeled_comments_ = labeled_comments_[labeled_comments_['sentiment_by_trained_model'].notna()]
labeled_comments_ = labeled_comments_.groupby("_id_y").filter(lambda x: len(x) > 5)
high_satisfaction_id = labeled_comments_.groupby('productId')['sentiment_by_trained_model'].agg(['mean', 'count']
                        ).sort_values(by=['mean', 'count'], ascending=[False, False]).head(1000).index.tolist()
high_satisfaction_data = labeled_comments_[labeled_comments_['_id_y'].isin(high_satisfaction_id)].drop_duplicates(subset='_id_y')
low_satisfaction_id = labeled_comments_.groupby('productId')['sentiment_by_trained_model'].agg(['mean', 'count']
                        ).sort_values(by=['mean', 'count'], ascending=[True, False]).head(1000).index.tolist()
low_satisfaction_data = labeled_comments_[labeled_comments_['_id_y'].isin(low_satisfaction_id)].drop_duplicates(subset='_id_y')
high_data = high_satisfaction_data.groupby('main_category')['_id_y'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
high_data['percent'] = round(high_data['count'] / sum(high_data['count']), 2)
low_data = low_satisfaction_data.groupby('main_category')['_id_y'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
low_data['percent'] = round(low_data['count'] / sum(low_data['count']), 2)


In [ ]:
all_products_data = products.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
all_products_data['percent'] = round(all_products_data['count'] / sum(all_products_data['count']), 3)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=all_products_data['main_category'],
    y=all_products_data['percent'],
    name='All Products',
    marker_color='blue',
    opacity=0.6,
    text=all_products_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=high_data['main_category'],
    y=high_data['percent'],
    name='High satisfaction Products',
    marker_color='green',
    opacity=0.6,
    text=high_data['count'], 
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=low_data['main_category'],
    y=low_data['percent'],
    name='Low satisfaction Products',
    marker_color='red',
    opacity=0.6,
    text=low_data['count'], 
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.update_layout(
    title='Percentage of Products by Main Category',
    xaxis_title='Main Category',
    yaxis_title='Percent',
    xaxis_tickangle=-30  
)
fig.show()

In [ ]:
fig = px.pie(high_data, values='percent', names='main_category', title='Percentage of High Satisfaction Products by Main Category', hover_data=['count'])
fig.show()

fig = px.pie(low_data, values='percent', names='main_category', title='Percentage of Low Satisfaction Products by Main Category', hover_data=['count'])
fig.show()

In [ ]:
products_by_comment = labeled_comments_.groupby(['productId','sentiment_by_parsbert']).size().unstack()
pos_products = products_by_comment[products_by_comment['Negative'] < products_by_comment['Positive']].reset_index()
neg_products = products_by_comment[products_by_comment['Negative'] > 5].reset_index()
pos_products.index.name = None
neg_products.index.name = None
pos_products = pd.merge(pos_products, products, left_on='productId', right_on='_id')
neg_products = pd.merge(neg_products, products, left_on='productId', right_on='_id')
high_data = pos_products.groupby('main_category').size().reset_index()
low_data = neg_products.groupby('main_category').size().reset_index()
high_data.columns  = ['main_category', 'count']
low_data.columns  = ['main_category', 'count']
high_data['percent'] = high_data['count']/sum(high_data['count'])
low_data['percent'] = low_data['count']/sum(low_data['count'])
low_data

In [ ]:
all_products_data = products.groupby('main_category')['_id'].agg(['count']).sort_values(by='count', ascending=False).reset_index()
all_products_data['percent'] = round(all_products_data['count'] / sum(all_products_data['count']), 3)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=all_products_data['main_category'],
    y=all_products_data['percent'],
    name='All Products',
    marker_color='blue',
    opacity=0.6,
    text=all_products_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=high_data['main_category'],
    y=high_data['percent'],
    name='High satisfaction Products',
    marker_color='green',
    opacity=0.6,
    text=high_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))
fig.add_trace(go.Bar(
    x=low_data['main_category'],
    y=low_data['percent'],
    name='Low satisfaction Products',
    marker_color='red',
    opacity=0.6,
    text=low_data['count'],  
    hovertemplate='Category: %{x}<br>Percent: %{y}<br>Count: %{text}<extra></extra>'
))

fig.update_layout(
    title='Percentage of Products by Main Category',
    xaxis_title='Main Category',
    yaxis_title='Percent',
    xaxis_tickangle=-30 
)

fig.show()

In [ ]:
fig = px.pie(high_data, values='percent', names='main_category', title='Percentage of high Satisfaction Products by Main Category', hover_data=['count'])
fig.show()

fig = px.pie(low_data, values='percent', names='main_category', title='Percentage of Low Satisfaction Products by Main Category', hover_data=['count'])
fig.show()